# PURITY Inference

Runs the unified config-driven inference pipeline for PURITY.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pyarrow.parquet as pq

from pioneerml.integration.zenml import load_step_output
from pioneerml.integration.zenml import utils as zenml_utils

PROJECT_ROOT = zenml_utils.setup_repo_pythonpath(Path(zenml_utils.find_project_root()).resolve())
from pioneerml_purity_plugin.purity.pipeline import inference_pipeline, load_config

zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)
sys.path.insert(0, str(PROJECT_ROOT / 'artifacts'))
from generate_purity_dummy_parquet import generate_dummy_purity_parquet


Using ZenML repository root: /workspace
Ensure this is the top-level of your repo (.zen must live here).


## Build Input + Resolve Model

In [2]:
input_parquet = PROJECT_ROOT / 'artifacts' / 'purity_notebook_inference.parquet'
generate_dummy_purity_parquet(output_path=input_parquet, num_events=32, seed=23)

model_path_file = PROJECT_ROOT / 'artifacts' / 'purity_small_torchscript_path.txt'
if model_path_file.exists():
    model_path = Path(model_path_file.read_text(encoding='utf-8').strip()).resolve()
else:
    candidates = sorted((PROJECT_ROOT / 'artifacts' / 'purity_notebook_export').glob('*_torchscript.pt'))
    if not candidates:
        raise FileNotFoundError('Run training notebook first to produce a PURITY torchscript export.')
    model_path = candidates[-1].resolve()

model_path


PosixPath('/workspace/artifacts/purity_small_export/purity_small_20260416_081507_20260416_081528_torchscript.pt')

## Patch Config and Run

In [3]:
cfg = load_config()['inference']
cfg['model_handle_builder']['model_handle']['type'] = 'torchscript'
cfg['model_handle_builder']['model_handle']['config']['model_path'] = str(model_path)
cfg['inference']['loader_manager']['config']['input_sources_spec']['main_sources'] = [str(input_parquet)]
cfg['inference']['loader_manager']['config']['input_sources_spec']['optional_sources_by_name'] = {}
cfg['inference']['loader_manager']['config']['input_sources_spec']['source_type'] = 'file'
cfg['inference']['writer']['config']['output_dir'] = str(PROJECT_ROOT / 'artifacts' / 'purity_notebook_predictions')
cfg['inference']['writer']['config']['fallback_output_dir'] = str(PROJECT_ROOT / 'artifacts' / 'purity_notebook_predictions')
cfg['inference']['writer']['config']['write_timestamped'] = False

run = inference_pipeline.with_options(enable_cache=False)(pipeline_config=cfg)
out = load_step_output(run, 'run_inference')
out


Initiating a new run for the pipeline: inference_pipeline.
Caching is disabled by default for inference_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step build_model_handle has started.
Step build_model_handle has finished in 0.165s.
Step run_inference has started.
Step run_inference has finished in 2.007s.
Pipeline run has finished in 3.050s.


{'predictions_path': '/workspace/artifacts/purity_notebook_predictions/purity_notebook_inference_preds.parquet',
 'predictions_paths': ['/workspace/artifacts/purity_notebook_predictions/purity_notebook_inference_preds.parquet'],
 'timestamped_predictions_path': None,
 'timestamped_predictions_paths': []}

In [4]:
pred_path = Path(load_step_output(run, 'run_inference')['predictions_path'])
tbl = pq.read_table(pred_path)
print(pred_path)
print(tbl.schema)
tbl.slice(0, 3).to_pydict()

/workspace/artifacts/purity_notebook_predictions/purity_notebook_inference_preds.parquet
event_id: int64
pred_purity_signal: list<element: float>
  child 0, element: float
pred_purity_logit: list<element: float>
  child 0, element: float
time_group_ids: list<element: int64>
  child 0, element: int64


{'event_id': [0, 1, 2],
 'pred_purity_signal': [[0.6397599577903748,
   0.6427042484283447,
   0.6434102654457092,
   0.6415327191352844,
   0.6421735882759094,
   0.6403059363365173,
   0.6439344882965088],
  [0.6445439457893372,
   0.580115020275116,
   0.6391535401344299,
   0.6473976969718933,
   0.5827274322509766,
   0.6473376154899597,
   0.6404135227203369],
  [0.5802525281906128,
   0.640565812587738,
   0.6437597274780273,
   0.5781302452087402,
   0.5501269698143005,
   0.6438168883323669,
   0.6444834470748901]],
 'pred_purity_logit': [[0.5743224024772644,
   0.5871206521987915,
   0.5901967287063599,
   0.5820227861404419,
   0.5848105549812317,
   0.5766920447349548,
   0.5924823880195618],
  [0.5951414108276367,
   0.3232455253601074,
   0.5716923475265503,
   0.607620120048523,
   0.3339799642562866,
   0.6073567867279053,
   0.5771595239639282],
  [0.3238101601600647,
   0.5778206586837769,
   0.5917202234268188,
   0.31510257720947266,
   0.20118382573127747,
   0.591